# Test Set Separation

Replica o mesmo split treino/teste usado pela Isa (HFUSBW.ipynb) para fixar o conjunto de teste em todos os experimentos de radiômica.

**Dataset fonte:** `data/raw/DATASET-ISA/CLASSES/` — subpastas `classe_b` (benigno) e `classe_m` (maligno), imagens BW nomeadas `{id}_bw.jpg`. As imagens Doppler correspondentes estão em `data/raw/archive/images/bw/` com o padrão `{id}_doppler.jpg`.

**Estratégia de split:** itera `os.listdir` sobre `DATASET-ISA/CLASSES/` (subpastas classe_b/classe_m), depois aplica `train_test_split(test_size=0.10, random_state=42, stratify=labels)` — idêntico ao notebook da Isa.

**Resultado esperado:** 20 imagens de teste fixas (13 benignas + 7 malignas) / 178 imagens trainval de 198 no total.

Saídas salvas em `data/images/sync-images/`:
- `test_set/test_ids.csv` — IDs e rótulos das imagens do conjunto de teste fixo
- `test_set/bw/` — imagens BW de teste
- `test_set/doppler/` — imagens Doppler de teste
- `trainval_set/bw/classe_b|classe_m/` — imagens BW de trainval
- `trainval_set/doppler/classe_b|classe_m/` — imagens Doppler de trainval

## Env Configuration

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED      = 42
TEST_SIZE = 0.10

# Mesmos caminhos que o notebook da Isa usa
BW_DIR      = os.path.join("..", "data", "raw", "DATASET-ISA", "CLASSES")
DOPPLER_DIR = os.path.join("..", "data", "raw", "archive", "images", "bw")

TEST_OUT     = os.path.join("..", "data", "images", "sync-images", "test_set")
TRAINVAL_OUT = os.path.join("..", "data", "images", "sync-images", "trainval_set")

for cls in ["classe_b", "classe_m"]:
    os.makedirs(os.path.join(TRAINVAL_OUT, "bw",      cls), exist_ok=True)
    os.makedirs(os.path.join(TRAINVAL_OUT, "doppler", cls), exist_ok=True)

os.makedirs(os.path.join(TEST_OUT, "bw"),      exist_ok=True)
os.makedirs(os.path.join(TEST_OUT, "doppler"), exist_ok=True)

## Functions

In [2]:
def load_paths_and_labels(image_dir, label_map):
    """Percorre as subpastas de classe e retorna arrays paralelos de caminhos e rótulos numéricos."""
    paths, labels = [], []
    for class_name in os.listdir(image_dir):
        class_path = os.path.join(image_dir, class_name)
        if os.path.isdir(class_path):
            for img_name in os.listdir(class_path):
                paths.append(os.path.join(class_path, img_name))
                labels.append(label_map[class_name])
    return np.array(paths), np.array(labels)

## Process

In [3]:
# Carrega caminhos BW — mesma ordem que o loop os.listdir da Isa
# classe_b → 0 (benigno), classe_m → 1 (maligno)
label_map = {"classe_b": 0, "classe_m": 1}
image_paths, labels_num = load_paths_and_labels(BW_DIR, label_map)

print(f"Total de amostras: {len(image_paths)}")
print(f"  Benigno  : {(labels_num == 0).sum()}")
print(f"  Maligno  : {(labels_num == 1).sum()}")

Total de amostras: 198
  Benigno  : 131
  Maligno  : 67


In [4]:
# Replica o split da Isa exatamente
idx = np.arange(len(image_paths))
idx_trainval, idx_test = train_test_split(
    idx, test_size=TEST_SIZE, random_state=SEED, stratify=labels_num
)

print(f"Teste    : {len(idx_test)} amostras  (benigno={(labels_num[idx_test]==0).sum()}, maligno={(labels_num[idx_test]==1).sum()})")
print(f"Trainval : {len(idx_trainval)} amostras")

Teste    : 20 amostras  (benigno=13, maligno=7)
Trainval : 178 amostras


In [5]:
# Salva test_ids.csv
# IDs extraídos do nome do arquivo: "{id}_bw.jpg"
test_image_ids = [
    os.path.basename(p).replace("_bw.jpg", "")
    for p in image_paths[idx_test]
]
label_str_map = {0: "BENIGN", 1: "MALIGNANT"}

df_ids = pd.DataFrame({
    "image_id": test_image_ids,
    "label":    [label_str_map[l] for l in labels_num[idx_test]],
    "target":   labels_num[idx_test],
}).sort_values("image_id").reset_index(drop=True)

df_ids.to_csv(os.path.join(TEST_OUT, "test_ids.csv"), index=False)
print(df_ids)
print(f"\nIDs de teste fixos: {sorted(test_image_ids)}")

   image_id      label  target
0       107     BENIGN       0
1       113     BENIGN       0
2       142  MALIGNANT       1
3       147  MALIGNANT       1
4       160  MALIGNANT       1
5       167  MALIGNANT       1
6       170  MALIGNANT       1
7       180  MALIGNANT       1
8       192     BENIGN       0
9        24     BENIGN       0
10       27     BENIGN       0
11       36     BENIGN       0
12       40     BENIGN       0
13       46     BENIGN       0
14       56     BENIGN       0
15       65     BENIGN       0
16       80     BENIGN       0
17       83     BENIGN       0
18       87  MALIGNANT       1
19       99     BENIGN       0

IDs de teste fixos: ['107', '113', '142', '147', '160', '167', '170', '180', '192', '24', '27', '36', '40', '46', '56', '65', '80', '83', '87', '99']


In [6]:
# Copia imagens BW de teste
bw_out = os.path.join(TEST_OUT, "bw")
for p in image_paths[idx_test]:
    shutil.copy2(p, bw_out)
print(f"Imagens BW de teste copiadas: {len(os.listdir(bw_out))}")

Imagens BW de teste copiadas: 20


In [7]:
# Copia imagens Doppler de teste usando os mesmos image_ids
# Doppler fica em DOPPLER_DIR com padrão "{id}_doppler.jpg" (pasta flat, sem subpastas de classe)
dop_out = os.path.join(TEST_OUT, "doppler")
test_ids_set = set(test_image_ids)

for fname in os.listdir(DOPPLER_DIR):
    if "_doppler" not in fname:
        continue
    img_id = fname.replace("_doppler.jpg", "")
    if img_id in test_ids_set:
        shutil.copy2(os.path.join(DOPPLER_DIR, fname), dop_out)

print(f"Imagens Doppler de teste copiadas: {len(os.listdir(dop_out))}")

Imagens Doppler de teste copiadas: 20


In [8]:
# Verifica estratificação
orig_rate = labels_num.mean()
test_rate = labels_num[idx_test].mean()
tv_rate   = labels_num[idx_trainval].mean()

print(f"Taxa de maligno — Total: {orig_rate:.3f} | Teste: {test_rate:.3f} | Trainval: {tv_rate:.3f}")
print("Estratificação OK" if abs(test_rate - orig_rate) < 0.05 else "ATENÇÃO: proporção de classes desviou")

Taxa de maligno — Total: 0.338 | Teste: 0.350 | Trainval: 0.337
Estratificação OK


In [9]:
# Copia imagens Doppler de trainval preservando subpastas de classe
# Precisa do mapa image_id → classe para colocar na subpasta correta
id_to_class = {}
for class_name in os.listdir(BW_DIR):
    class_path = os.path.join(BW_DIR, class_name)
    if not os.path.isdir(class_path):
        continue
    for fname in os.listdir(class_path):
        img_id = fname.replace("_bw.jpg", "")
        id_to_class[img_id] = class_name

trainval_ids = set(
    os.path.basename(image_paths[i]).replace("_bw.jpg", "")
    for i in idx_trainval
)

for fname in os.listdir(DOPPLER_DIR):
    if "_doppler" not in fname:
        continue
    img_id = fname.replace("_doppler.jpg", "")
    if img_id in trainval_ids:
        cls = id_to_class[img_id]
        shutil.copy2(os.path.join(DOPPLER_DIR, fname), os.path.join(TRAINVAL_OUT, "doppler", cls))

dop_ben = len(os.listdir(os.path.join(TRAINVAL_OUT, "doppler", "classe_b")))
dop_mal = len(os.listdir(os.path.join(TRAINVAL_OUT, "doppler", "classe_m")))
print(f"Doppler trainval — classe_b: {dop_ben}, classe_m: {dop_mal}, total: {dop_ben + dop_mal}")

Doppler trainval — classe_b: 118, classe_m: 60, total: 178


In [10]:
# Copia imagens BW de trainval preservando subpastas de classe
cls_map = {0: "classe_b", 1: "classe_m"}
for i in idx_trainval:
    cls = cls_map[labels_num[i]]
    shutil.copy2(image_paths[i], os.path.join(TRAINVAL_OUT, "bw", cls))

bw_ben = len(os.listdir(os.path.join(TRAINVAL_OUT, "bw", "classe_b")))
bw_mal = len(os.listdir(os.path.join(TRAINVAL_OUT, "bw", "classe_m")))
print(f"BW trainval — classe_b: {bw_ben}, classe_m: {bw_mal}, total: {bw_ben + bw_mal}")

BW trainval — classe_b: 118, classe_m: 60, total: 178
